# Task 6 - Write Paper and Final Report

This notebook is the writing and reporting workspace. It is designed to pull together:

- the motivation from the repo and README
- the experiment outputs from Tasks 2 through 5
- the W&B tables and charts
- the final paper outline and slide structure


## Repo Understanding

This project is a forked AssetOpsBench workflow for training Gemma to plan without tool descriptions.

- `src/servers/` contains the local MCP servers: `iot`, `fmsr`, `tsfm`, and `utilities`.
- `src/workflow/` contains the plan / execute runner used by the benchmark scripts.
- `benchmark/generate_data/datasets/` contains the three SFT datasets used by the current notebook:
  `tool_knowledge.jsonl`, `planning.jsonl`, and `execution.jsonl`.
- `benchmark/baseline_tests/gemini_flash_informed_results.json` is the main gold-plan file used by evaluation.
- `notebook/AssetOpsBench_Gemma_FineTuning.ipynb` is the current end-to-end Colab notebook.

Current notebook gaps that matter for the TODO list:

- training still uses `report_to="none"` in `SFTConfig`
- model loading still uses `torch_dtype=...` instead of `dtype=...`
- pipeline evaluation drops dependencies by hardcoding `Dependency: None`
- the specialist pipeline currently logs no latency analysis, confusion matrix, or W&B tables


In [ ]:
!pip install -U transformers peft trl accelerate bitsandbytes
!pip install -U datasets evaluate rouge-score pandas matplotlib seaborn scikit-learn
!pip install -U wandb sentencepiece protobuf


In [ ]:
import os
import re
import gc
import json
import math
import time
import random
import warnings
import inspect
from copy import deepcopy
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import wandb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu:", torch.cuda.get_device_name(0), f"({props.total_memory / 1e9:.1f} GB)")


In [ ]:
RESULTS_ROOT = Path("/content/output_full_run")

def maybe_load_json(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

def maybe_load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

experiment_json = maybe_load_json(RESULTS_ROOT / "results.json")
per_scenario_df = maybe_load_csv(RESULTS_ROOT / "per_scenario.csv")
print("results keys:", list(experiment_json.keys()) if experiment_json else [])
print("per_scenario shape:", per_scenario_df.shape)


## Paper Outline

Use the cells below as a paper-first workspace. The structure matches the TODO list:

1. introduction
2. related work
3. method
4. experiments
5. results
6. discussion
7. production argument
8. presentation slides


In [ ]:
            paper_sections = {
                "introduction": '''
The current AssetOpsBench plan call pays a large prompt tax because the planner receives tool
descriptions every time. In this repo, the README frames that tax at roughly 2.3k tokens per query.
Our goal is to internalize tool knowledge inside a small Gemma model so that blind-mode planning
becomes viable while preserving routing quality and argument quality.
''',
                "related_work": '''
Connect this project to:
- AssetOpsBench and MCP-style tool planning
- tool-use LLM evaluation
- QLoRA and PEFT for parameter-efficient adaptation
- modular or specialist architectures for tool routing
''',
                "method": '''
Describe the three-tier architecture already reflected in the current notebook:
- generalist model trained on all examples
- planner model trained on routing-only outputs
- specialist models trained on per-agent argument generation

Also document the training data sources:
- tool knowledge dataset
- planning dataset
- execution dataset
''',
                "experiments": '''
Summarize:
- baseline blind vs informed
- generalist fine-tuning
- planner fine-tuning
- specialist pipeline
- hyperparameter sweeps
- OOD evaluation on expanded configs
''',
                "results": '''
Call out:
- AT-F1
- ArgKey and ArgVal
- dependency correctness
- token savings
- latency tradeoffs for parallel specialists
''',
                "discussion": '''
Discuss:
- what improved after fine-tuning
- where the WorkOrder alignment problem still limits clean comparison
- why single-agent and multi-agent results differ
- what the failure cases say about routing vs argument generation
''',
                "production_argument": '''
Argue for a planner + parallel specialists system:
- smaller routing model handles plan structure
- specialists can run in parallel once routing is known
- token savings compound because tool manuals are not repeated
- failure modes are easier to isolate per domain
''',
            }

            for name, text in paper_sections.items():
                print("\n" + "=" * 80)
                print(name.upper())
                print("=" * 80)
                print(text.strip())


In [ ]:
if not per_scenario_df.empty:
    summary_tables = {
        "per_type": per_scenario_df.groupby("type").mean(numeric_only=True).round(3),
        "top_pipeline_cases": per_scenario_df.sort_values("pipeline_atf1", ascending=False).head(10),
        "hard_cases": per_scenario_df.sort_values("pipeline_atf1", ascending=True).head(10),
    }
    for name, table in summary_tables.items():
        print("\nTABLE:", name)
        display(table)


In [ ]:
slide_outline = [
    "Problem: MCP planning has a repeated tool-description token overhead",
    "Benchmark: AssetOpsBench scenarios and domain agents",
    "Method: generalist vs planner + specialists",
    "Training data and no-leakage split",
    "Main results: baseline vs fine-tuned",
    "Specialist vs generalist analysis",
    "Hyperparameter sweep takeaways",
    "Failure cases and WorkOrder mismatch",
    "Production architecture argument",
    "Next steps and risks",
]

for idx, item in enumerate(slide_outline, start=1):
    print(f"{idx}. {item}")


In [ ]:
            abstract_draft = f"""We study whether a small Gemma model can internalize MCP tool knowledge for AssetOpsBench-style planning,
reducing the need to repeat tool descriptions in every prompt. Using a three-tier architecture with a
generalist model, a routing planner, and domain specialists, we evaluate blind-mode planning quality,
argument fidelity, dependency correctness, and token savings. The final report should populate this
abstract with the measured gains from the full run, sweep, and specialist-vs-generalist analysis.
"""

            print(abstract_draft)
